# Croatian deployment test: XLM-R pipeline on hand-curated thoughts

Mirrors `tqa_pr_pipeline_arch_deploy_test.ipynb` (Chapter 4.6.7 deployment validation) with three changes:

1. **Models**: `xlm-roberta-base` checkpoints trained in Stage 4 on translated TherapistQA (binary) and translated PatternReframe (10-way, α = 0.3).
2. **Inputs**: same 30 deployment-realistic thoughts as the English notebook, **hand-translated to Croatian** to avoid translationese. The Croatian translations are not from any test set.
3. **Outputs**: suffixed with `_hr`.

The hand-curated 30 examples (10 obviously distorted / 10 non-distorted / 10 ambiguous) are the same intents as Chapter 4.6.7, allowing direct EN-vs-HR comparison of pipeline behaviour on production-realistic input. This is the experimental basis for the language-mode architecture decision in Chapter 6.

In [1]:
# === Setup ===
from pathlib import Path

MODELS = Path("../../models")
BINARY_CHECKPOINT       = MODELS / "05_01_cro_therapistqa_binary_xlm" / "model"
PR_10_WAY_CHECKPOINT    = MODELS / "05_03_cro_patternreframe_10_way_xlm" / "model"

RESULTS_DIR = Path("./results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Binary checkpoint: {BINARY_CHECKPOINT}")
print(f"Type   checkpoint: {PR_10_WAY_CHECKPOINT}")
print(f"Binary exists:     {BINARY_CHECKPOINT.exists()}")
print(f"Type   exists:     {PR_10_WAY_CHECKPOINT.exists()}")

Binary checkpoint: ../../models/05_01_cro_therapistqa_binary_xlm/model
Type   checkpoint: ../../models/05_03_cro_patternreframe_10_way_xlm/model
Binary exists:     True
Type   exists:     True


In [2]:
# Each tuple is (intent, Croatian thought, expected English-label name).

EXAMPLES_HR = [
    # ===== Obviously distorted (one per category) =====
    # All-or-nothing
    ("distorted", "Ako ne dobijem ovo promaknuće, cijela mi je karijera promašaj.",
     "All-or-nothing thinking"),
    # Catastrophizing
    ("distorted", "Pogriješio sam jednu stvar u prezentaciji i sada će svi misliti da sam nesposoban.",
     "Catastrophizing"),
    # Discounting the positive
    ("distorted", "Rekla je da joj se sviđa moj rad, ali sigurno je samo bila pristojna.",
     "Discounting the positive"),
    # Fortune-telling
    ("distorted", "Jednostavno znam da će ovaj spoj proći loše.",
     "Fortune-telling"),
    # Mind reading
    ("distorted", "Nije mi odgovorio na poruku, sigurno je ljut na mene.",
     "Mind reading"),
    # Labeling
    ("distorted", "Zaboravio sam poslati email. Ja sam takav idiot.",
     "Labeling"),
    # Mental filter
    ("distorted", "Svi su pljeskali osim jedne osobe koja je izgledala kao da joj je dosadno. Cijelo izlaganje je bila katastrofa.",
     "Mental filtering"),
    # Overgeneralization
    ("distorted", "Upropastio sam ovaj razgovor za posao. Nikada se neću zaposliti.",
     "Overgeneralization"),
    # Personalization
    ("distorted", "Partner mi je loše raspoložen. Sigurno sam ja nešto napravio.",
     "Personalization"),
    # Should statements
    ("distorted", "Trebao bih biti puno dalje u karijeri u ovim godinama.",
     "Should statements"),

    # ===== Non-distorted (10) =====
    ("nondistorted", "Trebao bih nazvati mamu ovaj vikend.", "No Distortion"),
    ("nondistorted", "Izgleda da bi moglo kišiti kasnije.", "No Distortion"),
    ("nondistorted", "Sutra u 15 sati imam sastanak.", "No Distortion"),
    ("nondistorted", "Kava je danas dobra.", "No Distortion"),
    ("nondistorted", "Želim ove godine više čitati.", "No Distortion"),
    ("nondistorted", "Bole me leđa nakon cijelog dana sjedenja.", "No Distortion"),
    ("nondistorted", "Umoran sam i mislim da ću ranije u krevet.", "No Distortion"),
    ("nondistorted", "Rok je u petak pa moram završiti izvještaj do četvrtka.", "No Distortion"),
    ("nondistorted", "Nisam se složio s onim što je rekla, ali razumijem njezin stav.", "No Distortion"),
    ("nondistorted", "Danas sam napredovao, iako još ima posla.", "No Distortion"),

    # ===== Ambiguous / borderline (10) =====
    # Could be a fact or a distortion depending on intent
    ("ambiguous", "Nije me pozdravio na sastanku.", "uncertain"),
    # Genuine concern vs catastrophizing
    ("ambiguous", "Brinem se da neću imati dovoljno novca za mirovinu.", "uncertain"),
    # Realistic expectation vs all-or-nothing
    ("ambiguous", "Ako sredstva ne stignu, projekt je gotov.", "uncertain"),
    # Reasonable inference vs mind reading
    ("ambiguous", "Šef me ovog tjedna izbjegavao pogledom.", "uncertain"),
    # Self-criticism that may be deserved
    ("ambiguous", "Nisam smio izgubiti živce s njim.", "uncertain"),
    # Reasonable concern vs fortune-telling
    ("ambiguous", "S obzirom na trenutno tržište, bit će teško naći posao u mojoj struci.", "uncertain"),
    # Mixed positive and negative
    ("ambiguous", "Općenito mi je dobro išlo, ali stalno razmišljam o onom jednom lošem odgovoru.", "uncertain"),
    # Reasonable doubt
    ("ambiguous", "Mislim da nisam dovoljno kvalificiran za ovu poziciju.", "uncertain"),
    # Self-blame that may or may not be appropriate
    ("ambiguous", "Tim nije stigao do roka, a ja sam bila voditeljica.", "uncertain"),
    # General malaise
    ("ambiguous", "Nešto ne valja, ali ne mogu točno odrediti što.", "uncertain"),
]

print(f"Total examples: {len(EXAMPLES_HR)}")
print(f"  Obviously distorted:    {sum(1 for x in EXAMPLES_HR if x[0] == 'distorted')}")
print(f"  Obviously non-distorted: {sum(1 for x in EXAMPLES_HR if x[0] == 'nondistorted')}")
print(f"  Ambiguous / borderline:  {sum(1 for x in EXAMPLES_HR if x[0] == 'ambiguous')}")

Total examples: 30
  Obviously distorted:    10
  Obviously non-distorted: 10
  Ambiguous / borderline:  10


In [3]:
# Load both XLM-R Croatian models from disk
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.nn.functional import softmax

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Binary detector (XLM-R, trained on translated TherapistQA)
print(f"\nLoading binary detector from {BINARY_CHECKPOINT}")
tok_b = AutoTokenizer.from_pretrained(BINARY_CHECKPOINT)
mdl_b = AutoModelForSequenceClassification.from_pretrained(BINARY_CHECKPOINT)
mdl_b.to(device).eval()
cfg_b = [mdl_b.config.id2label[i] for i in sorted(mdl_b.config.id2label, key=int)]
print(f"  Binary detector labels: {cfg_b}")
assert cfg_b == ["No Distortion", "Distorted"], f"Unexpected binary labels: {cfg_b}"

# 10-way type classifier (XLM-R, trained on translated PatternReframe, soft-label α=0.3)
print(f"\nLoading 10-way type classifier from {PR_10_WAY_CHECKPOINT}")
tok_t = AutoTokenizer.from_pretrained(PR_10_WAY_CHECKPOINT)
mdl_t = AutoModelForSequenceClassification.from_pretrained(PR_10_WAY_CHECKPOINT)
mdl_t.to(device).eval()
cfg_t = [mdl_t.config.id2label[i] for i in sorted(mdl_t.config.id2label, key=int)]
print(f"  Type classifier labels (10-way): {cfg_t}")
assert len(cfg_t) == 10, f"Expected 10 distortion classes, found {len(cfg_t)}"

# Sanity check: both classifiers' labels must be the same English category set.
# (We did NOT translate labels — they stayed in English in the bilingual CSVs.)
print("\nNote: model output labels are in English. The classifier was trained")
print("on Croatian text → English category names, since categories are language-agnostic.")

/Users/ivagoluza/Documents/cognitive-distortioins-ml/dataset_preparation/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Device: cpu

Loading binary detector from ../../models/05_01_cro_therapistqa_binary_xlm/model
  Binary detector labels: ['No Distortion', 'Distorted']

Loading 10-way type classifier from ../../models/05_03_cro_patternreframe_10_way_xlm/model
  Type classifier labels (10-way): ['Black-and-white or polarized thinking / All or nothing thinking', 'Catastrophizing', 'Discounting the positive', 'Jumping to conclusions: Fortune-telling', 'Jumping to conclusions: mind reading', 'Labeling and mislabeling', 'Mental filtering', 'Overgeneralization', 'Personalization', 'Should statements']

Note: model output labels are in English. The classifier was trained
on Croatian text → English category names, since categories are language-agnostic.


In [4]:
MAX_LEN = 512


def predict_one(text, tokenizer, model):
    """Return the full probability distribution for a single input."""
    enc = tokenizer(text, truncation=True, max_length=MAX_LEN,
                    return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(**enc).logits
    return softmax(logits, dim=-1).cpu().numpy()[0]


# Run both models on all 30 Croatian examples
binary_probs = []
type_argmax  = []
type_max_prob = []

for intent, text, _ in EXAMPLES_HR:
    # Binary detector
    pb = predict_one(text, tok_b, mdl_b)
    binary_probs.append(float(pb[1]))   # prob(distorted)

    # Type classifier
    pt = predict_one(text, tok_t, mdl_t)
    type_argmax.append(cfg_t[int(np.argmax(pt))])
    type_max_prob.append(float(np.max(pt)))

print(f"Done. {len(EXAMPLES_HR)} Croatian examples processed.")

Done. 30 Croatian examples processed.


In [5]:
import pandas as pd

# Pipeline rule (same as English): if binary says ND (prob_distorted < 0.5),
# output "No Distortion"; otherwise, output the type classifier's argmax.
pipeline_preds = []
binary_calls   = []
for i in range(len(EXAMPLES_HR)):
    is_distorted = binary_probs[i] >= 0.5
    binary_calls.append("Distorted" if is_distorted else "No Distortion")
    pipeline_preds.append(type_argmax[i] if is_distorted else "No Distortion")

EXPECTED_TO_LONG = {
    "All-or-nothing thinking":  "Black-and-white or polarized thinking / All or nothing thinking",
    "Catastrophizing":          "Catastrophizing",
    "Discounting the positive": "Discounting the positive",
    "Fortune-telling":          "Jumping to conclusions: Fortune-telling",
    "Mind reading":             "Jumping to conclusions: mind reading",
    "Labeling":                 "Labeling and mislabeling",
    "Mental filtering":         "Mental filtering",
    "Overgeneralization":       "Overgeneralization",
    "Personalization":          "Personalization",
    "Should statements":        "Should statements",
    "No Distortion":            "No Distortion",
    "uncertain":                "uncertain",
}

SHORT_LABEL = {
    "No Distortion":                                                    "ND",
    "Black-and-white or polarized thinking / All or nothing thinking":  "All-or-nothing",
    "Catastrophizing":                                                  "Catastrophizing",
    "Discounting the positive":                                         "Discounting +",
    "Jumping to conclusions: Fortune-telling":                          "Fortune-telling",
    "Jumping to conclusions: mind reading":                             "Mind reading",
    "Labeling and mislabeling":                                         "Labeling",
    "Mental filtering":                                                 "Mental filter",
    "Overgeneralization":                                               "Overgeneralization",
    "Personalization":                                                  "Personalization",
    "Should statements":                                                "Should",
    "uncertain":                                                        "(ambiguous)",
}

rows = []
for i, (intent, text, expected) in enumerate(EXAMPLES_HR):
    long_expected = EXPECTED_TO_LONG[expected]
    if expected == "uncertain":
        agreement = "ambiguous"
    elif pipeline_preds[i] == long_expected:
        agreement = "✓ match"
    else:
        agreement = "✗ different"

    rows.append({
        "Intent":              intent,
        "Thought":             text if len(text) <= 70 else text[:67] + "...",
        "Expected":            SHORT_LABEL[long_expected],
        "Binary call":         binary_calls[i],
        "Bin prob(D)":         f"{binary_probs[i]:.2f}",
        "Pipeline pred":       SHORT_LABEL[pipeline_preds[i]],
        "Type argmax":         SHORT_LABEL[type_argmax[i]],
        "Type max-prob":       f"{type_max_prob[i]:.2f}",
        "Agreement":           agreement,
    })

results = pd.DataFrame(rows)
print(results.to_string(index=False))
results.to_csv(RESULTS_DIR / "deployment_examples_pipeline_hr.csv", index=False)
print(f"\nSaved to {RESULTS_DIR / 'deployment_examples_pipeline_hr.csv'}")

      Intent                                                                Thought           Expected   Binary call Bin prob(D)   Pipeline pred        Type argmax Type max-prob   Agreement
   distorted         Ako ne dobijem ovo promaknuće, cijela mi je karijera promašaj.     All-or-nothing     Distorted        0.97  All-or-nothing     All-or-nothing          0.66     ✓ match
   distorted Pogriješio sam jednu stvar u prezentaciji i sada će svi misliti da ...    Catastrophizing     Distorted        0.97    Mind reading       Mind reading          0.53 ✗ different
   distorted  Rekla je da joj se sviđa moj rad, ali sigurno je samo bila pristojna.      Discounting + No Distortion        0.04              ND      Discounting +          0.50 ✗ different
   distorted                           Jednostavno znam da će ovaj spoj proći loše.    Fortune-telling     Distorted        0.93 Fortune-telling    Fortune-telling          0.57     ✓ match
   distorted                  Nije mi odgovorio na

In [6]:
print("=== Summary by intent (Croatian pipeline) ===\n")

for intent in ["distorted", "nondistorted", "ambiguous"]:
    sub_idx = [i for i, ex in enumerate(EXAMPLES_HR) if ex[0] == intent]
    n = len(sub_idx)

    if intent == "distorted":
        matches = sum(
            1 for i in sub_idx
            if pipeline_preds[i] == EXPECTED_TO_LONG[EXAMPLES_HR[i][2]]
        )
        bin_caught = sum(1 for i in sub_idx if binary_probs[i] >= 0.5)

        print(f"DISTORTED ({n} examples):")
        print(f"  Binary stage flagged as distorted: {bin_caught}/{n}")
        print(f"  Pipeline matched expected category: {matches}/{n}")
        missed = n - bin_caught
        if missed:
            print(f"  ⚠ {missed} distorted thoughts missed by binary stage:")
            for i in sub_idx:
                if binary_probs[i] < 0.5:
                    print(f"     - prob(D)={binary_probs[i]:.2f}: \"{EXAMPLES_HR[i][1][:60]}...\"")
        print()

    elif intent == "nondistorted":
        correct_nd = sum(1 for i in sub_idx if pipeline_preds[i] == "No Distortion")
        bin_correct_nd = sum(1 for i in sub_idx if binary_probs[i] < 0.5)

        print(f"NONDISTORTED ({n} examples):")
        print(f"  Binary stage correctly said 'No Distortion': {bin_correct_nd}/{n}")
        print(f"  Pipeline correctly output 'No Distortion':   {correct_nd}/{n}")
        fp_idx = [i for i in sub_idx if binary_probs[i] >= 0.5]
        if fp_idx:
            print(f"  ⚠ {len(fp_idx)} neutral thoughts misclassified as distorted:")
            for i in fp_idx:
                print(f"     - prob(D)={binary_probs[i]:.2f}, type→{SHORT_LABEL[type_argmax[i]]}: "
                      f"\"{EXAMPLES_HR[i][1][:50]}...\"")
        print()

    else:  # ambiguous
        nd_pipeline = sum(1 for i in sub_idx if pipeline_preds[i] == "No Distortion")
        avg_bin_prob = np.mean([binary_probs[i] for i in sub_idx])
        avg_type_max = np.mean([type_max_prob[i] for i in sub_idx])

        print(f"AMBIGUOUS ({n} examples):")
        print(f"  Pipeline output 'No Distortion': {nd_pipeline}/{n}")
        print(f"  Mean binary prob(distorted):     {avg_bin_prob:.3f}")
        print(f"  Mean type-classifier max-prob:   {avg_type_max:.3f}")
        print()

# Confidence tier breakdown of binary probabilities
print("=== Binary stage confidence distribution ===")
for lo, hi, name in [
    (0.0, 0.3, "high confidence ND"),
    (0.3, 0.5, "leaning ND but uncertain"),
    (0.5, 0.7, "leaning distorted but uncertain"),
    (0.7, 1.0, "high confidence distorted"),
]:
    count = sum(1 for p in binary_probs if lo <= p < hi)
    print(f"  prob(D) ∈ [{lo:.1f}, {hi:.1f}) — {name}: {count} examples")

=== Summary by intent (Croatian pipeline) ===

DISTORTED (10 examples):
  Binary stage flagged as distorted: 3/10
  Pipeline matched expected category: 2/10
  ⚠ 7 distorted thoughts missed by binary stage:
     - prob(D)=0.04: "Rekla je da joj se sviđa moj rad, ali sigurno je samo bila p..."
     - prob(D)=0.04: "Nije mi odgovorio na poruku, sigurno je ljut na mene...."
     - prob(D)=0.20: "Zaboravio sam poslati email. Ja sam takav idiot...."
     - prob(D)=0.40: "Svi su pljeskali osim jedne osobe koja je izgledala kao da j..."
     - prob(D)=0.14: "Upropastio sam ovaj razgovor za posao. Nikada se neću zaposl..."
     - prob(D)=0.24: "Partner mi je loše raspoložen. Sigurno sam ja nešto napravio..."
     - prob(D)=0.06: "Trebao bih biti puno dalje u karijeri u ovim godinama...."

NONDISTORTED (10 examples):
  Binary stage correctly said 'No Distortion': 10/10
  Pipeline correctly output 'No Distortion':   10/10

AMBIGUOUS (10 examples):
  Pipeline output 'No Distortion': 6/10
  Mean bi

## Direct EN vs HR comparison

If the English deployment results CSV exists at the path below, compute paired per-row agreement: same intent, same expected category — does the English pipeline and the Croatian pipeline produce the same final prediction?

In [7]:
EN_RESULTS_PATH = MODELS / "04_03_architecture_experiments/results/deployment_examples_pipeline.csv"

if not EN_RESULTS_PATH.exists():
    print(f"English deployment results not found at {EN_RESULTS_PATH}")
    print("Skip this cell or place the English CSV at that path.")
else:
    en = pd.read_csv(EN_RESULTS_PATH)
    hr = results  # the HR result DataFrame from the cell above

    if len(en) != len(hr):
        print(f"Row counts differ: EN={len(en)}, HR={len(hr)}. Cannot compare row-wise.")
    else:
        en["hr_pipeline_pred"] = hr["Pipeline pred"].values
        en["hr_binary_prob"]   = hr["Bin prob(D)"].values
        en["hr_type_max_prob"] = hr["Type max-prob"].values
        en["hr_intent"]        = hr["Intent"].values
        en["same_pred"]        = (en["Pipeline pred"] == en["hr_pipeline_pred"])

        for intent in ["distorted", "nondistorted", "ambiguous"]:
            sub = en[en["Intent"] == intent]
            agree = int(sub["same_pred"].sum())
            print(f"  {intent:<14}  EN-HR pipeline agreement: {agree}/{len(sub)}")

        # Where they disagreed
        diff = en[~en["same_pred"]][["Intent", "Thought", "Expected", "Pipeline pred", "hr_pipeline_pred"]]
        diff.columns = ["Intent", "Thought (EN)", "Expected", "EN pred", "HR pred"]
        if len(diff):
            print(f"\nDisagreements ({len(diff)}):")
            print(diff.to_string(index=False))

        en.to_csv(RESULTS_DIR / "deployment_examples_pipeline_en_vs_hr.csv", index=False)
        print(f"\nSaved paired comparison to {RESULTS_DIR / 'deployment_examples_pipeline_en_vs_hr.csv'}")

  distorted       EN-HR pipeline agreement: 4/10
  nondistorted    EN-HR pipeline agreement: 10/10
  ambiguous       EN-HR pipeline agreement: 7/10

Disagreements (9):
   Intent                                                           Thought (EN)           Expected            EN pred         HR pred
distorted I made one mistake in the presentation and now everyone will think ...    Catastrophizing Overgeneralization    Mind reading
distorted                   He didn't text me back, so he must be angry with me.       Mind reading       Mind reading              ND
distorted                         I forgot to send the email. I'm such an idiot.           Labeling           Labeling              ND
distorted Everyone clapped except for one person who looked bored. The whole ...      Mental filter     All-or-nothing              ND
distorted             I bombed this job interview. I'm never going to find work. Overgeneralization Overgeneralization              ND
distorted             